
# TASK 2 – Retrieval-Augmented Generation (RAG) from Scratch

## 1. Objective

The objective of this task is to build a question-answering system over a public document using Retrieval-Augmented Generation (RAG).

The system extracts information from a PDF, divides the text into smaller chunks, converts the chunks into embeddings, and retrieves relevant passages based on a user's question.

The retrieved passages are then provided to a language model to generate an answer.

This project uses Python, Sentence Transformers, NumPy, FAISS and an open-weight language model without using LangChain or LlamaIndex.


In [ ]:

# Install the libraries required for our RAG project

# !pip install -q pymupdf sentence-transformers faiss-cpu transformers accelerate bitsandbytes


In [ ]:

import os
import re
import requests
import fitz  # PyMuPDF
import numpy as np
import faiss
import torch

from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM

print("All libraries imported successfully!")
print("GPU available:", torch.cuda.is_available())


/usr/local/lib/python3.13/dist-packages/torch/library.py:508: UserWarning: Warning only once for all operators,  other operators may also be overridden.
  Overriding a previously registered kernel for the same operator and the same dispatch key
  operator: bitsandbytes::int8_mm_dequant(Tensor A, Tensor row_stats, Tensor col_stats, ScalarType? dtype=None, Tensor? bias=None) -> Tensor
    registered at /usr/local/lib/python3.13/dist-packages/torch/library.py:750
  dispatch key: CompositeExplicitAutograd
  previous kernel: no debug info
       new kernel: registered at /usr/local/lib/python3.13/dist-packages/torch/library.py:940 (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/core/dispatch/OperatorEntry.cpp:208.)
  self.m.impl(


All libraries imported successfully!
GPU available: False



## 2. Loading the Public PDF Document

We use the NIST Artificial Intelligence Risk Management Framework (AI RMF 1.0), a publicly available document.

The PDF will be downloaded and its text will be extracted for our RAG system.


In [ ]:

# Download the NIST PDF

pdf_url = "https://nvlpubs.nist.gov/nistpubs/ai/NIST.AI.100-1.pdf"
pdf_path = "NIST_AI_RMF.pdf"

response = requests.get(pdf_url)
response.raise_for_status()

with open(pdf_path, "wb") as file:
    file.write(response.content)

print("PDF downloaded successfully!")
print("File size:", round(os.path.getsize(pdf_path) / (1024 * 1024), 2), "MB")


PDF downloaded successfully!
File size: 1.86 MB


In [ ]:

# Open the downloaded PDF

document = fitz.open(pdf_path)

print("Total number of pages:", len(document))

# Extract text from every page
pages = []

for page_number, page in enumerate(document):
    text = page.get_text()

    if text.strip():
        pages.append({
            "page_number": page_number + 1,
            "text": text
        })

print("Pages with extracted text:", len(pages))
print("\nFirst page preview:\n")
print(pages[0]["text"][:1500])


Total number of pages: 48
Pages with extracted text: 48

First page preview:

NIST AI 100-1
Artificial Intelligence Risk Management
Framework (AI RMF 1.0)



In [ ]:

total_characters = sum(len(page["text"]) for page in pages)

print("Total extracted characters:", total_characters)
print("Total extracted pages:", len(pages))


Total extracted characters: 106478
Total extracted pages: 48



## 3. Text Chunking

Chunking is the process of dividing a large document into smaller pieces of text.

We create two different chunk configurations:

1. 200 words per chunk with 40 words overlap.
2. 400 words per chunk with 40 words overlap.

We will compare both configurations later to identify which provides better retrieval accuracy.


In [ ]:

# Function to split text into chunks

def create_chunks(pages, chunk_size, overlap):
    chunks = []

    for page in pages:
        words = page["text"].split()

        start = 0

        while start < len(words):
            end = start + chunk_size

            chunk_text = " ".join(words[start:end])

            if chunk_text.strip():
                chunks.append({
                    "text": chunk_text,
                    "page_number": page["page_number"]
                })

            start += chunk_size - overlap

    return chunks


# Create chunks with two different sizes

chunks_200 = create_chunks(
    pages,
    chunk_size=200,
    overlap=40
)

chunks_400 = create_chunks(
    pages,
    chunk_size=400,
    overlap=40
)

print("200-word chunks:", len(chunks_200))
print("400-word chunks:", len(chunks_400))

print("\nFirst 200-word chunk:\n")
print(chunks_200[0]["text"][:1000])


200-word chunks: 121
400-word chunks: 72

First 200-word chunk:

NIST AI 100-1 Artificial Intelligence Risk Management Framework (AI RMF 1.0)



## 4. Generating Text Embeddings

We use the Sentence Transformer model all-MiniLM-L6-v2 to convert document chunks into numerical vectors.

These vectors help the system identify passages that are semantically similar to a user's question.


In [ ]:

# Load the embedding model

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

# Generate embeddings for both chunk configurations

texts_200 = [chunk["text"] for chunk in chunks_200]
texts_400 = [chunk["text"] for chunk in chunks_400]

embeddings_200 = embedding_model.encode(
    texts_200,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

embeddings_400 = embedding_model.encode(
    texts_400,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

embeddings_200 = embeddings_200.astype("float32")
embeddings_400 = embeddings_400.astype("float32")

print("200-word embedding shape:", embeddings_200.shape)
print("400-word embedding shape:", embeddings_400.shape)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (6 outputs left out)

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

200-word embedding shape: (121, 384)
400-word embedding shape: (72, 384)



## 5. Vector Database Using FAISS

We use FAISS to store and search the generated embeddings.

The vectors are normalized, so inner-product similarity is equivalent to cosine similarity.


In [ ]:

def create_faiss_index(embeddings):
    dimension = embeddings.shape[1]

    index = faiss.IndexFlatIP(dimension)
    index.add(embeddings)

    return index


index_200 = create_faiss_index(embeddings_200)
index_400 = create_faiss_index(embeddings_400)

print("FAISS indexes created successfully!")
print("200-word index size:", index_200.ntotal)
print("400-word index size:", index_400.ntotal)


FAISS indexes created successfully!
200-word index size: 121
400-word index size: 72



## 6. Test Questions and Ground-Truth Passages

We prepare 15 questions based on the NIST AI Risk Management Framework.

For each question, we identify a supporting passage and its page number from the original document. These passages act as ground truth for evaluating retrieval.


In [ ]:

test_questions = [
    {"question": "What is the main purpose of the NIST AI Risk Management Framework?", "anchor": "AI Risk Management Framework"},
    {"question": "Who are the intended users of the AI RMF?", "anchor": "AI actors"},
    {"question": "What is the purpose of the GOVERN function?", "anchor": "GOVERN"},
    {"question": "What does the MAP function focus on?", "anchor": "MAP"},
    {"question": "What is the purpose of the MEASURE function?", "anchor": "MEASURE"},
    {"question": "What does the MANAGE function help organizations do?", "anchor": "MANAGE"},
    {"question": "What does valid and reliable AI mean?", "anchor": "valid and reliable"},
    {"question": "Why is safety important in trustworthy AI?", "anchor": "safe"},
    {"question": "What does secure and resilient AI mean?", "anchor": "secure and resilient"},
    {"question": "What is meant by accountable and transparent AI?", "anchor": "accountable and transparent"},
    {"question": "What is the difference between explainability and interpretability?", "anchor": "explainable and interpretable"},
    {"question": "How does privacy contribute to trustworthy AI?", "anchor": "privacy"},
    {"question": "How should fairness be considered in AI systems?", "anchor": "fair"},
    {"question": "How does the framework address harmful bias?", "anchor": "harmful bias"},
    {"question": "What is risk tolerance in AI risk management?", "anchor": "risk tolerance"}
]

# Find a supporting passage in the original PDF text

def find_source_passage(anchor):
    candidates = []

    for page in pages:
        page_text = " ".join(page["text"].split())
        position = page_text.lower().find(anchor.lower())

        if position != -1 and page["page_number"] > 5:
            count = page_text.lower().count(anchor.lower())
            candidates.append((count, page["page_number"], page_text, position))

    if not candidates:
        raise ValueError(
            f"Anchor not found: {anchor}. Choose another phrase."
        )

    # Select the page containing the most occurrences
    _, page_number, page_text, position = max(
        candidates, key=lambda item: item[0]
    )

    start = max(0, position - 300)
    end = min(len(page_text), position + 500)

    return {
        "page_number": page_number,
        "source_passage": page_text[start:end]
    }


# Attach source passages to all questions

for item in test_questions:
    source = find_source_passage(item["anchor"])
    item.update(source)

print("Total test questions:", len(test_questions))

for i, item in enumerate(test_questions, 1):
    print(f"\nQuestion {i}: {item['question']}")
    print("Source page:", item["page_number"])
    print("Supporting passage:", item["source_passage"])


Total test questions: 15

Question 1: What is the main purpose of the NIST AI Risk Management Framework?
Source page: 8
Supporting passage: ns at mul- tiple public forums, and many small group meetings – has informed development of the AI RMF 1.0 as well as AI research and development and evaluation conducted by NIST and others. Priority research and additional guidance that will enhance this Framework will be captured in an associated AI Risk Management Framework Roadmap to which NIST and the broader community can contribute. Page 3

Question 2: Who are the intended users of the AI RMF?
Source page: 14
Supporting passage: ng the AI RMF may face different challenges than large organizations, depending on their capabilities and resources. 2. Audience Identifying and managing AI risks and potential impacts – both positive and negative – re- quires a broad set of perspectives and actors across the AI lifecycle. Ideally, AI actors will represent a diversity of experience, expertise, and ba


## 7. Semantic Retrieval

When a user asks a question, it is converted into an embedding using the same embedding model.

FAISS compares the question embedding with document chunk embeddings and retrieves the five most relevant passages.


In [ ]:

def retrieve_chunks(question, index, chunks, top_k=5):
    query_embedding = embedding_model.encode(
        [question],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    scores, indices = index.search(query_embedding, top_k)

    results = []

    for score, idx in zip(scores[0], indices[0]):
        if idx == -1:
            continue

        results.append({
            "text": chunks[idx]["text"],
            "page_number": chunks[idx]["page_number"],
            "score": float(score)
        })

    return results


# Test retrieval with one question

sample_question = test_questions[0]["question"]

sample_results = retrieve_chunks(
    sample_question,
    index_200,
    chunks_200
)

print("Question:", sample_question)

for i, result in enumerate(sample_results, 1):
    print(f"\nResult {i} | Page {result['page_number']}")
    print("Similarity:", round(result["score"], 4))
    print(result["text"][:400])


Question: What is the main purpose of the NIST AI Risk Management Framework?

Result 1 | Page 1
Similarity: 0.7687
NIST AI 100-1 Artificial Intelligence Risk Management Framework (AI RMF 1.0)

Result 2 | Page 2
Similarity: 0.7594
NIST AI 100-1 Artificial Intelligence Risk Management Framework (AI RMF 1.0) This publication is available free of charge from: https://doi.org/10.6028/NIST.AI.100-1 January 2023 U.S. Department of Commerce Gina M. Raimondo, Secretary National Institute of Standards and Technology Laurie E. Locascio, NIST Director and Under Secretary of Commerce for Standards and Technology

Result 3 | Page 6
Similarity: 0.7273
NIST AI 100-1 AI RMF 1.0 Executive Summary Artificial intelligence (AI) technologies have significant potential to transform society and people’s lives – from commerce and health to transportation and cybersecurity to the envi- ronment and our planet. AI technologies can drive inclusive economic growth and support scientific advancements that improve th


## 8. Retrieval Evaluation – Recall@5

Recall@5 measures whether the relevant source passage is found among the top five retrieved chunks.

We evaluate both 200-word and 400-word chunk configurations using the same 15 questions and their supporting passages.

Recall@5 = Number of questions with a relevant passage in top 5 / Total questions


In [ ]:

stop_words = {
    "the", "and", "for", "are", "that", "this", "with",
    "from", "into", "what", "which", "their", "they",
    "have", "has", "was", "were", "can", "how", "does",
    "its", "who", "why", "not", "but", "also", "such",
    "these", "those", "will", "should", "about"
}


def content_words(text):
    words = set(re.findall(r"[a-z]{3,}", text.lower()))
    return words - stop_words


def evaluate_retrieval(index, chunks):
    rows = []

    for item in test_questions:
        results = retrieve_chunks(
            item["question"], index, chunks, top_k=5
        )

        gold_words = content_words(item["source_passage"])

        best_overlap = 0
        hit = False

        for result in results:
            chunk_words = content_words(result["text"])

            if not gold_words:
                continue

            overlap = len(gold_words & chunk_words) / len(gold_words)
            best_overlap = max(best_overlap, overlap)

            relevant = (
                result["page_number"] == item["page_number"]
                and overlap >= 0.10
            )

            if relevant:
                hit = True

        rows.append({
            "question": item["question"],
            "gold_page": item["page_number"],
            "hit": hit,
            "best_overlap": best_overlap,
            "top_results": results
        })

    recall = sum(row["hit"] for row in rows) / len(rows)

    return recall, rows


recall_200, results_200 = evaluate_retrieval(
    index_200, chunks_200
)

recall_400, results_400 = evaluate_retrieval(
    index_400, chunks_400
)

print("RETRIEVAL EVALUATION")
print("--------------------")
print(f"Recall@5 (200-word chunks): {recall_200 * 100:.2f}%")
print(f"Recall@5 (400-word chunks): {recall_400 * 100:.2f}%")

if recall_200 > recall_400:
    better_chunk_size = 200
elif recall_400 > recall_200:
    better_chunk_size = 400
else:
    better_chunk_size = 200  # Tie-breaker for demonstration

print("\nSelected chunk size:", better_chunk_size, "words")


RETRIEVAL EVALUATION
--------------------
Recall@5 (200-word chunks): 66.67%
Recall@5 (400-word chunks): 46.67%

Selected chunk size: 200 words


In [ ]:

# Use the better-performing chunk configuration

if better_chunk_size == 200:
    evaluation_rows = results_200
else:
    evaluation_rows = results_400


failures = [row for row in evaluation_rows if not row["hit"]]

print("Actual retrieval failures:", len(failures))

if len(failures) >= 2:
    selected_cases = failures[:2]
elif len(failures) == 1:
    selected_cases = failures + sorted(
        [r for r in evaluation_rows if r["hit"]],
        key=lambda x: x["best_overlap"]
    )[:1]
else:
    selected_cases = sorted(
        evaluation_rows,
        key=lambda x: x["best_overlap"]
    )[:2]

for i, row in enumerate(selected_cases, 1):
    print(f"\nCASE {i}")
    print("Question:", row["question"])
    print("Gold page:", row["gold_page"])
    print("Gold passage overlap:", round(row["best_overlap"], 3))
    print("Retrieved passages:")

    for result in row["top_results"][:3]:
        print(
            f"Page {result['page_number']} | "
            f"Score {result['score']:.3f}"
        )
        print(result["text"][:300])


Actual retrieval failures: 5

CASE 1
Question: Who are the intended users of the AI RMF?
Gold page: 14
Gold passage overlap: 0.242
Retrieved passages:
Page 7 | Score 0.642
of the AI RMF is to offer a resource to the organizations designing, developing, deploying, or using AI systems to help manage the many risks of AI and promote trustwor- thy and responsible development and use of AI systems. The Framework is intended to be voluntary, rights-preserving, non-sector-sp
Page 26 | Score 0.628
NIST AI 100-1 AI RMF 1.0 deployed, or evaluated – which can create opportunities to surface problems and identify existing and emergent risks. An online companion resource to the AI RMF, the NIST AI RMF Playbook, is available to help organizations navigate the AI RMF and achieve its outcomes through
Page 47 | Score 0.625
NIST AI 100-1 AI RMF 1.0 Appendix D: Attributes of the AI RMF NIST described several key attributes of the AI RMF when work on the Framework first began. These attributes have remain


## 9. Retrieval Failure Analysis

### Failure Case 1
**Question:** [Insert the question]

**Problem:** The relevant source passage was not found in the top five retrieved chunks.

**Possible reason:** The question may use different words from the source document, or the relevant information may be split across chunks.

**Possible improvement:** Use hybrid search combining semantic embeddings with BM25 keyword search, and experiment with chunk size and overlap.

### Failure Case 2
**Question:** [Insert the question]

**Problem:** [Describe what the retrieval output showed.]

**Possible reason:** [Explain based on the retrieved passages.]

**Possible improvement:** Use a re-ranker to reorder retrieved passages based on their relevance to the question.



## 10. Answer Generation Using an LLM

We use Qwen2.5-0.5B-Instruct as the language model.

The retrieved passages are provided as context to the model. The model generates an answer based on the information retrieved from the PDF.


In [ ]:

# Load the Qwen language model

llm_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(llm_name)

model = AutoModelForCausalLM.from_pretrained(
    llm_name,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

model.eval()

print("Qwen model loaded successfully!")


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Qwen model loaded successfully!


In [ ]:

# Select the better-performing retrieval index

if better_chunk_size == 200:
    final_index = index_200
    final_chunks = chunks_200
else:
    final_index = index_400
    final_chunks = chunks_400


def rag_answer(question):

    # Step 1: Retrieve relevant passages
    retrieved = retrieve_chunks(
        question,
        final_index,
        final_chunks,
        top_k=5
    )

    # Step 2: Prepare context
    context = ""

    for i, item in enumerate(retrieved, 1):
        context += (
            f"\nPassage {i} (Page {item['page_number']}):\n"
            f"{item['text']}\n"
        )

    # Step 3: Create the prompt
    messages = [
        {
            "role": "system",
            "content": (
                "You are a question-answering assistant. "
                "Answer using only the provided document context. "
                "If the answer is not available in the context, "
                "say that the document does not provide enough information. "
                "Do not invent facts."
            )
        },
        {
            "role": "user",
            "content": (
                f"Context:\n{context}\n\n"
                f"Question: {question}\n\n"
                "Give a clear and concise answer."
            )
        }
    ]

    # Step 4: Generate answer
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=180,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    answer = tokenizer.decode(
        output[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    return answer, retrieved


In [ ]:

# Ask a question about the NIST document

question = "What are the four main functions of the NIST AI RMF?"

answer, retrieved_passages = rag_answer(question)

print("USER QUESTION:")
print(question)

print("\nGENERATED ANSWER:")
print(answer)

print("\nSOURCES:")
for item in retrieved_passages:
    print(
        f"Page {item['page_number']} "
        f"(Similarity: {item['score']:.4f})"
    )


USER QUESTION:
What are the four main functions of the NIST AI RMF?

GENERATED ANSWER:
The four main functions of the NIST AI RMF are:
1. GOVERN - This function aims to establish guidelines and policies for governing AI systems.
2. MAP - This function focuses on mapping out AI systems to ensure they meet certain criteria.
3. MEASURE - This function involves assessing the effectiveness of AI systems against defined metrics.
4. MANAGE - This function oversees the implementation and maintenance of AI systems to ensure they remain effective and aligned with organizational goals.

SOURCES:
Page 25 (Similarity: 0.6881)
Page 8 (Similarity: 0.6251)
Page 26 (Similarity: 0.6212)
Page 38 (Similarity: 0.6107)
Page 6 (Similarity: 0.5958)



## 11. BM25 and Re-ranker

### What does BM25 add to embedding search?

Embedding search finds passages based on semantic meaning. However, it may sometimes miss exact keywords, technical terms, names, or abbreviations.

BM25 is a keyword-based retrieval technique. It searches for matching words and considers how important and frequent those words are in a document.

Combining BM25 with embedding search is called hybrid search. It can improve retrieval by considering both meaning and exact keyword matches.

### What does a Re-ranker do?

A re-ranker is a model that takes the question and initially retrieved passages and evaluates how relevant each passage is to the question.

It then rearranges the passages from most relevant to least relevant.

In simple words, retrieval finds possible answers, while a re-ranker helps select the best passages from those results.


In [ ]:

print("=" * 50)
print("FINAL RAG RETRIEVAL RESULTS")
print("=" * 50)

print("Total test questions:", len(test_questions))

print(
    f"200-word chunk Recall@5: "
    f"{recall_200 * 100:.2f}%"
)

print(
    f"400-word chunk Recall@5: "
    f"{recall_400 * 100:.2f}%"
)

if recall_200 > recall_400:
    print("Better configuration: 200-word chunks")
elif recall_400 > recall_200:
    print("Better configuration: 400-word chunks")
else:
    print("Both configurations achieved equal Recall@5.")

print("\nQuestion-wise retrieval results:")

for i, row in enumerate(results_200, 1):
    print(
        f"{i}. {'PASS' if row['hit'] else 'FAIL'} - "
        f"{row['question']}"
    )

print("\nRAG system is ready!")


FINAL RAG RETRIEVAL RESULTS
Total test questions: 15
200-word chunk Recall@5: 66.67%
400-word chunk Recall@5: 46.67%
Better configuration: 200-word chunks

Question-wise retrieval results:
1. PASS - What is the main purpose of the NIST AI Risk Management Framework?
2. FAIL - Who are the intended users of the AI RMF?
3. PASS - What is the purpose of the GOVERN function?
4. PASS - What does the MAP function focus on?
5. FAIL - What is the purpose of the MEASURE function?
6. PASS - What does the MANAGE function help organizations do?
7. FAIL - What does valid and reliable AI mean?
8. PASS - Why is safety important in trustworthy AI?
9. FAIL - What does secure and resilient AI mean?
10. FAIL - What is meant by accountable and transparent AI?
11. PASS - What is the difference between explainability and interpretability?
12. PASS - How does privacy contribute to trustworthy AI?
13. PASS - How should fairness be considered in AI systems?
14. PASS - How does the framework address harmful bias?

In [ ]:

import json

report_data = {
    "document": "NIST AI Risk Management Framework",
    "total_pages": len(pages),
    "total_questions": len(test_questions),
    "chunk_sizes": [200, 400],
    "overlap": 40,
    "recall_at_5": {
        "200_word_chunks": recall_200,
        "400_word_chunks": recall_400
    },
    "better_chunk_size": better_chunk_size,
    "test_questions": test_questions
}

with open("task2_results.json", "w", encoding="utf-8") as file:
    json.dump(report_data, file, indent=4, ensure_ascii=False)

print("Results saved successfully!")


Results saved successfully!


In [ ]:

from google.colab import files

files.download("task2_results.json")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>